In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd


## Diferencia entre archivos `.conn` y `.bsa`

### `.conn`

Archivo de resultados de conectividad exportado por **BESA Connectivity**.

Es el archivo principal que vamos a importar para estadística.

Contiene los valores numéricos de conectividad para una combinación concreta de:

- banda de frecuencia
- método de conectividad
- método time-frequency previo
- condición experimental
- pares canal-canal o fuente-fuente
- latencias y frecuencias

En este script, los `.conn` serán la fuente principal de datos.

---

### `.bsa`

Archivo de solución BESA.

No es la tabla principal de conectividad.

Puede contener información asociada a las fuentes, modelos espaciales o soluciones usadas por BESA.

En esta fase no vamos a importar los `.bsa` para estadística.

Los conservamos como archivos auxiliares porque podrían ser útiles más adelante para interpretar fuentes, montajes o metadatos del análisis.

## Funciones principales

`read_single_conn_file()` lee un archivo `.conn` de BESA Connectivity y extrae la cabecera, las fuentes/canales, los tiempos, las frecuencias y la matriz de conectividad en formato `source × target × frequency × time`.

`conn_matrix_to_long_df()` convierte esa matriz en un `DataFrame` long, donde cada fila corresponde a un valor de conectividad para un sujeto, condición, frecuencia, tiempo y par `source-target`.

`clean_conn_long_df()` depura el `DataFrame`: elimina las conexiones diagonales (`source == target`) y conserva solo las frecuencias que pertenecen a la banda original del análisis, por ejemplo `alpha = 8–13 Hz`. Permite decidir si los rangos de frecuencia son inclusivos (`[f_min, f_max]`) o semiabiertos (`[f_min, f_max)`).

`changing_columns()` añade columnas descriptivas a partir de la columna `condition`.

La función usa un diccionario definido por el usuario. Cada dimensión del diccionario corresponde a una nueva columna. El orden de las dimensiones en el diccionario define qué dígito de `condition` se usa para cada columna.

Por ejemplo, si `condition = "14"`:

- la primera dimensión usa `"1"`
- la segunda dimensión usa `"4"`

`select_values_df()` selecciona los valores finales que se usarán para estadística. Puede dejar los datos sin resumir, promediar la conectividad entre bins de frecuencia (`average_bin_frequencies=True), filtrar una ventana temporal (`select_time=(inicio_ms, fin_ms)`) y/o promediar los valores de conectividad en el tiempo (`average_over_time=True`).

Cuando se promedian frecuencias:

- `frequency_idx = -1`
- `frequency_hz = NaN`
- `frequency_summary = "average"`

Además, se añaden columnas resumen #del rango real de frecuencias promediado:

- `frequency_average_min_hz`
- `frequency_average_max_hz`
- `frequency_average_n_bins`

Cuando se promedia tiempo:

- `time_idx = -1`
- `time_ms = NaN`
- `time_summary = "average"`


## Ejemplos de uso

```python
# Mantener todos los valores disponibles
df_selected = select_values_df(df_conn_clean)
```

```python
# Promediar la conectividad entre los bins de frecuencia disponibles
df_selected = select_values_df(
    df_conn_clean,
    average_bin_frequencies=True
)
```

```python
# Filtrar una ventana temporal, manteniendo cada tiempo dentro de la ventana
df_selected = select_values_df(
    df_conn_clean,
    select_time=(0, 500)
)
```

```python
# Filtrar una ventana temporal y promediar conectividad en el tiempo
df_selected = select_values_df(
    df_conn_clean,
    select_time=(0, 500),
    average_over_time=True
)
```

```python
# Promediar frecuencias y después promediar tiempo en una ventana temporal
df_selected = select_values_df(
    df_conn_clean,
    average_bin_frequencies=True,
    select_time=(0, 500),
    average_over_time=True
)
```

In [ ]:
def read_single_conn_file(conn_file):
    """
    Lee un único archivo .conn de BESA Connectivity.

    Estructura BESA .conn:
    - línea 1: cabecera con metadatos
    - línea 2: nombres de canales/fuentes
    - resto: bloques numéricos separados por líneas vacías

    Cada bloque representa una matriz:
        frecuencia x tiempo

    La matriz final queda organizada como:
        source x target x frequency x time
    """

    conn_file = Path(conn_file)

    with open(conn_file, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    # ------------------------------------------------------------
    # 1. Cabecera y etiquetas
    # ------------------------------------------------------------

    header_line = lines[0].strip()
    channels = lines[1].strip().split()

    header_keys = [
    "VersionNumber",
    "DataType",
    "DecompositionType",
    "ConditionName",
    "NumberTrials",
    "NumberTimeSamples",
    "TimeStartInMS",
    "IntervalInMS",
    "NumberFrequencies",
    "FreqStartInHz",
    "FreqIntervalInHz",   # nombre observado en tus .conn exportados
    "FreqIntervalHz",     # variante documentada / posible en otros exports
    "Frequencies",
    "NumberChannels",
    ]

    # Parseo robusto de la cabecera.
    # Evita confundir "Frequencies=" con "NumberFrequencies=".
    pattern = r"(?<!\w)(" + "|".join(header_keys) + r")="

    matches = list(re.finditer(pattern, header_line))

    header = {}

    for i, match in enumerate(matches):
        key = match.group(1)
        value_start = match.end()

        if i + 1 < len(matches):
            value_end = matches[i + 1].start()
        else:
            value_end = len(header_line)

        header[key] = header_line[value_start:value_end].strip()

    # ------------------------------------------------------------
    # 2. Conversión de metadatos
    # ------------------------------------------------------------

    header["NumberTrials"] = int(header["NumberTrials"])
    header["NumberTimeSamples"] = int(header["NumberTimeSamples"])
    header["NumberFrequencies"] = int(header["NumberFrequencies"])
    header["NumberChannels"] = int(header["NumberChannels"])

    header["TimeStartInMS"] = float(header["TimeStartInMS"])
    header["IntervalInMS"] = float(header["IntervalInMS"])
    header["FreqStartInHz"] = float(header["FreqStartInHz"])

    n_channels = header["NumberChannels"]
    n_frequencies = header["NumberFrequencies"]
    n_times = header["NumberTimeSamples"]

    # ------------------------------------------------------------
    # 3. Vector temporal
    # ------------------------------------------------------------

    times = (
        header["TimeStartInMS"]
        + np.arange(n_times) * header["IntervalInMS"]
    )

    # ------------------------------------------------------------
    # 4. Vector de frecuencias
    # ------------------------------------------------------------

    if header["DecompositionType"].strip().lower() == "complex demodulation":
        # En Complex Demodulation se reconstruye el eje con inicio + intervalo.

        freq_interval_key = None

        if (
            "FreqIntervalInHz" in header
            and header["FreqIntervalInHz"].strip() != ""
        ):
            freq_interval_key = "FreqIntervalInHz"   # nombre observado en tus .conn

        elif (
            "FreqIntervalHz" in header
            and header["FreqIntervalHz"].strip() != ""
        ):
            freq_interval_key = "FreqIntervalHz"     # variante documentada / alternativa

        if freq_interval_key is None:
            raise ValueError(
                "No se encontró intervalo de frecuencia válido "
                "en FreqIntervalInHz ni en FreqIntervalHz."
            )

        header[freq_interval_key] = float(header[freq_interval_key])

        frequencies = (
            header["FreqStartInHz"]
            + np.arange(n_frequencies) * header[freq_interval_key]
        )

    else:
        # En Wavelet Transform / Morlet, BESA escribe las frecuencias explícitas.
        # En estos archivos FreqIntervalInHz puede venir vacío, por eso no se usa.
        frequencies = header["Frequencies"].split(";")
        frequencies = [f for f in frequencies if f.strip() != ""]
        frequencies = np.array(frequencies, dtype=float)

    if len(frequencies) != n_frequencies:
        raise ValueError(
            f"Número de frecuencias incorrecto: "
            f"{len(frequencies)} leídas, {n_frequencies} esperadas."
        )

    # ------------------------------------------------------------
    # 5. Separar los bloques numéricos
    # ------------------------------------------------------------

    data_lines = lines[2:]

    blocks = []
    current_block = []

    for line in data_lines:
        line = line.strip()

        if line == "":
            if current_block:
                blocks.append(current_block)
                current_block = []
        else:
            current_block.append(line)

    if current_block:
        blocks.append(current_block)

    expected_blocks = n_channels ** 2

    if len(blocks) != expected_blocks:
        raise ValueError(
            f"Número de bloques incorrecto: "
            f"{len(blocks)} encontrados, {expected_blocks} esperados."
        )

    # ------------------------------------------------------------
    # 6. Convertir bloques a matriz 4D
    # ------------------------------------------------------------

    conn_matrix = np.zeros(
        (n_channels, n_channels, n_frequencies, n_times),
        dtype=float
    )

    block_idx = 0

    for source_idx in range(n_channels):
        for target_idx in range(n_channels):

            block_array = np.array(
                [[float(x) for x in row.split()] for row in blocks[block_idx]],
                dtype=float
            )

            if block_array.shape != (n_frequencies, n_times):
                raise ValueError(
                    f"Bloque {block_idx} con forma {block_array.shape}; "
                    f"se esperaba {(n_frequencies, n_times)}."
                )

            conn_matrix[source_idx, target_idx, :, :] = block_array
            block_idx += 1

    # ------------------------------------------------------------
    # 7. Comprobación final de etiquetas
    # ------------------------------------------------------------

    if len(channels) != n_channels:
        raise ValueError(
            f"Número de etiquetas distinto de NumberChannels: "
            f"{len(channels)} etiquetas frente a {n_channels} canales/fuentes."
        )

    return header, channels, times, frequencies, conn_matrix

In [ ]:
def conn_matrix_to_long_df(
    conn_file,
    header,
    channels,
    times,
    frequencies,
    conn_matrix,
    freq_original_folder,
):
    """
    Convierte una matriz .conn de BESA a formato long.

    Cada fila representa un valor de conectividad para:
        subject x condition x frequency x time x source x target

    freq_original_folder:
        nombre de la carpeta original, por ejemplo 'cohalpha'.

    freq_original:
        banda original extraída desde freq_original_folder.
        Ejemplo: 'cohalpha' -> 'alpha'.
    """

    conn_file = Path(conn_file)

    subject_match = re.match(
    r"^(sub-[A-Za-z0-9]+|subject[A-Za-z0-9]+|s\d+[A-Za-z]?|\d+)_",
    conn_file.stem,
    re.IGNORECASE)    # acepta IDs tipo sub-01, subject01, s01/s01a o 001 antes del primer "_"

    if subject_match is None:
        subject = conn_file.stem
    else:
        subject = subject_match.group(1)

    # Extraer banda original desde el nombre de carpeta.
    # Ejemplo:
    #   cohalpha -> alpha
    #   cohbeta  -> beta
    #   plvtheta -> theta
    freq_original = re.sub(
        r"^(coh|dpli|wpli|plv|icoh|granger|pdc|dtf)",
        "",
        freq_original_folder,
        flags=re.IGNORECASE,
    )

    rows = []

    for source_idx, source in enumerate(channels):
        for target_idx, target in enumerate(channels):
            for freq_idx, frequency_hz in enumerate(frequencies):
                for time_idx, time_ms in enumerate(times):

                    rows.append({
                        "subject": subject,
                        "file": conn_file.name,

                        "freq_original_folder": freq_original_folder,
                        "freq_original": freq_original,

                        "connectivity_method": header["DataType"],
                        "tf_method": header["DecompositionType"],

                        "condition": header["ConditionName"],
                        "n_trials": header["NumberTrials"],

                        "frequency_idx": freq_idx,
                        "frequency_hz": frequency_hz,

                        "time_idx": time_idx,
                        "time_ms": time_ms,

                        "source": source,
                        "target": target,

                        "connectivity": conn_matrix[
                            source_idx,
                            target_idx,
                            freq_idx,
                            time_idx
                        ],
                    })

    return pd.DataFrame(rows)

In [ ]:
def clean_conn_long_df(df, frequency_ranges, inclusive_bands=True):
    """
    Limpia el DataFrame long de conectividad.

    Operaciones:
    1. Elimina conexiones diagonales: source == target.
    2. Elimina conexiones que involucren nodos Noise.
    3. Conserva solo las filas cuya frequency_hz cae dentro del rango
    definido para freq_original.

    inclusive_bands:
        True  -> usa intervalos cerrados [f_min, f_max].
        False -> usa intervalos semiabiertos [f_min, f_max), evitando solapamiento entre bandas.
    """

    df_clean = df.copy()

    # Eliminar diagonal
    df_clean = df_clean[df_clean["source"] != df_clean["target"]]


    # Eliminar conexiones que involucren Noise
    noise_mask = (
        df_clean["source"].astype(str).str.contains("Noise", case=False, na=False)
        | df_clean["target"].astype(str).str.contains("Noise", case=False, na=False)
    )

    df_clean = df_clean[~noise_mask]

    # Normalizar nombres de banda por seguridad
    df_clean["freq_original"] = (
        df_clean["freq_original"]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    # Comprobar que todas las bandas presentes tienen rango definido
    missing_bands = set(df_clean["freq_original"].unique()) - set(frequency_ranges.keys())

    if missing_bands:
        raise ValueError(
            f"Hay bandas sin rango definido en frequency_ranges: {missing_bands}"
        )

    # Filtrar por rango de frecuencia según freq_original
    keep_mask = pd.Series(False, index=df_clean.index)

    for band, (f_min, f_max) in frequency_ranges.items():

        if inclusive_bands:
            freq_mask = (
                (df_clean["frequency_hz"] >= f_min)
                & (df_clean["frequency_hz"] <= f_max)  # [f_min, f_max]: incluye borde superior
            )
        else:
            freq_mask = (
                (df_clean["frequency_hz"] >= f_min)
                & (df_clean["frequency_hz"] < f_max)   # [f_min, f_max): evita solapamiento entre bandas
            )

        band_mask = (
            (df_clean["freq_original"] == band)
            & freq_mask
        )

        keep_mask = keep_mask | band_mask

    df_clean = df_clean[keep_mask].reset_index(drop=True)

    return df_clean

In [ ]:
def reorder_connectivity_columns(df):
    """
    Reordena columnas finales manteniendo juntas las partes principales
    del pipeline: sujeto/archivo, banda/métodos, condición, frecuencia,
    tiempo, conexión y valor.

    Las columnas condition_* generadas por changing_columns() se colocan
    automáticamente después de condition.
    """

    cols = list(df.columns)

    condition_extra_cols = [
        col for col in cols
        if col.startswith("condition_")
    ]

    preferred_order = [
        "subject",
        "file",

        "freq_original_folder",
        "freq_original",

        "connectivity_method",
        "tf_method",

        "condition",
        *condition_extra_cols,

        "n_trials",

        "frequency_idx",
        "frequency_hz",
        "frequency_summary",
        "frequency_average_min_hz",
        "frequency_average_max_hz",
        "frequency_average_n_bins",

        "time_idx",
        "time_ms",
        "time_summary",

        "source",
        "target",

        "connectivity",
    ]

    ordered_cols = [
        col for col in preferred_order
        if col in cols
    ]

    remaining_cols = [
        col for col in cols
        if col not in ordered_cols
    ]

    return df[ordered_cols + remaining_cols]

In [ ]:
def average_values_df(
    df,
    average_bin_frequencies=True,
    select_time=False,
    average_over_time=False,
):
    """
    Selecciona o resume valores de conectividad.

    average_bin_frequencies:
        False      -> no promedia frecuencias.
        True  -> promedia connectivity entre las frecuencias disponibles
                      que llegan a esta función.

    select_time:
        False              -> no filtra tiempo.
        (start_ms, end_ms) -> conserva solo time_ms dentro de ese intervalo.

    average_over_time:
        False -> mantiene cada punto temporal tras aplicar select_time.
        True  -> promedia connectivity sobre los tiempos disponibles tras select_time.

    Nota:
        Las columnas derivadas de condition ya deben haber sido creadas antes
        por changing_columns(). Esta función respeta ese orden.
    """

    df_selected = df.copy()

    # ------------------------------------------------------------
    # 1. Selección temporal opcional
    # ------------------------------------------------------------

    if select_time is not False:
        time_start, time_end = select_time

        df_selected = df_selected[
            (df_selected["time_ms"] >= time_start)
            & (df_selected["time_ms"] <= time_end)
        ].copy()

    # ------------------------------------------------------------
    # 2. Selección / reducción en frecuencia
    # ------------------------------------------------------------

    if average_bin_frequencies is False:
        df_freq_selected = df_selected.copy()

    elif average_bin_frequencies == True:

        # Rango real de frecuencias que entra en el promedio.
        # En este pipeline, clean_conn_long_df() ya filtró previamente por banda.
        freq_summary_df = (
            df_selected
            .groupby(["subject", "file", "freq_original"], as_index=False)
            .agg(
                frequency_average_min_hz=("frequency_hz", "min"),
                frequency_average_max_hz=("frequency_hz", "max"),
                frequency_average_n_bins=("frequency_hz", "nunique"),
            )
        )

        group_cols = [
            col for col in df_selected.columns
            if col not in ["frequency_idx", "frequency_hz", "connectivity"]
        ]

        df_freq_selected = (
            df_selected
            .groupby(group_cols, as_index=False)
            .agg(connectivity=("connectivity", "mean"))
        )

        df_freq_selected["frequency_idx"] = -1        # frecuencia resumida
        df_freq_selected["frequency_hz"] = np.nan     # sin frecuencia única tras promediar
        df_freq_selected["frequency_summary"] = "average"

        df_freq_selected = df_freq_selected.merge(
            freq_summary_df,
            on=["subject", "file", "freq_original"],
            how="left"
        )

    else:
        raise ValueError(
            "average_bin_frequencies debe ser False o 'average'."
        )

    # ------------------------------------------------------------
    # 3. Promedio temporal opcional
    # ------------------------------------------------------------

    if average_over_time is False:

        df_freq_selected = reorder_connectivity_columns(df_freq_selected)

        return df_freq_selected.reset_index(drop=True)

    if average_bin_frequencies == True:
        group_cols = [
            col for col in df_freq_selected.columns
            if col not in [
                "time_idx",
                "time_ms",
                "connectivity",
                "frequency_idx",
                "frequency_hz",
            ]
        ]
    else:
        group_cols = [
            col for col in df_freq_selected.columns
            if col not in [
                "time_idx",
                "time_ms",
                "connectivity",
            ]
        ]

    df_time_selected = (
        df_freq_selected
        .groupby(group_cols, as_index=False)
        .agg(connectivity=("connectivity", "mean"))
    )

    df_time_selected["time_idx"] = -1              # tiempo resumido
    df_time_selected["time_ms"] = np.nan           # sin latencia única tras promediar
    df_time_selected["time_summary"] = "average"

    if average_bin_frequencies == True:
        df_time_selected["frequency_idx"] = -1     # frecuencia resumida
        df_time_selected["frequency_hz"] = np.nan  # sin frecuencia única tras promediar

    df_time_selected = reorder_connectivity_columns(df_time_selected)

    return df_time_selected.reset_index(drop=True)

In [ ]:
def changing_columns(df, condition_mapping=False):
    """
    Crea columnas derivadas a partir de la columna condition.

    condition_mapping:
        False -> no modifica el DataFrame
        dict  -> crea una columna por cada dimensión definida.

    El orden de las dimensiones en el diccionario define qué dígito de
    condition se usa para cada nueva columna.

    Ejemplo:
        condition = "14"

        primera dimensión  -> condition_self      -> usa "1" -> self
        segunda dimensión -> condition_emotional -> usa "4" -> positive
    """

    df_changed = df.copy()

    if condition_mapping is False:
        return df_changed

    df_changed["condition"] = df_changed["condition"].astype(str)

    n_dimensions = len(condition_mapping)

    # Comprobar que todos los códigos tienen longitud suficiente
    invalid_conditions = df_changed.loc[
        df_changed["condition"].str.len() < n_dimensions,
        "condition"
    ].unique()

    if len(invalid_conditions) > 0:
        raise ValueError(
            f"Hay condiciones con menos dígitos que dimensiones definidas: "
            f"{invalid_conditions}"
        )

    new_columns = []

    # Crear una columna nueva por cada dimensión, sin guardar columnas de código
    for digit_idx, (new_column, mapping) in enumerate(condition_mapping.items()):

        condition_codes = df_changed["condition"].str[digit_idx]
        df_changed[new_column] = condition_codes.map(mapping)

        missing_codes = condition_codes[df_changed[new_column].isna()].unique()

        if len(missing_codes) > 0:
            raise ValueError(
                f"Códigos no reconocidos para {new_column}: {missing_codes}"
            )

        new_columns.append(new_column)

    # Reordenar columnas: las nuevas columnas van justo después de condition
    cols = list(df_changed.columns)

    for col in new_columns:
        cols.remove(col)

    condition_pos = cols.index("condition")

    cols = (
        cols[:condition_pos + 1]
        + new_columns
        + cols[condition_pos + 1:]
    )

    df_changed = df_changed[cols]

    return df_changed





# Configuraciones y diccionarios 
Estos objetos se pueden modificar libremente en función de las necesidades experimentales

In [ ]:
# ============================================================
# clean_conn_long_df()
# ============================================================
# frequency_ranges:
#     dict -> rangos de frecuencia por banda.
#
# inclusive_bands:
#     True  -> incluye borde superior [f_min, f_max].
#     False -> excluye borde superior [f_min, f_max).

frequency_ranges = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "betalow": (13, 20),
    "betahigh": (20, 30),
    "gamma": (30, 40),
}

inclusive_bands = True


# ============================================================
# changing_columns()
# ============================================================
# condition_mapping:
#     False -> no añade columnas.
#     dict  -> crea columnas nuevas a partir de los dígitos de condition.

condition_mapping = {
    "condition_self": {
        "1": "self",
        "2": "self",
        "3": "self",
        "4": "friend",
        "5": "friend",
        "6": "friend",
        "7": "unknown",
        "8": "unknown",
        "9": "unknown",
    },

    "condition_emotional": {
        "4": "positive",
        "5": "neutral",
        "6": "negative",
    },
}

# condition_mapping = False


# ============================================================
# select_values_df()
# ============================================================
# average_bin_frequencies:
#     False     -> mantiene cada frecuencia.
#     True -> promedia frecuencias.
#
# select_time:
#     False              -> mantiene todos los tiempos.
#     (start_ms, end_ms) -> filtra una ventana temporal.
#
# average_over_time:
#     False -> mantiene cada tiempo.
#     True  -> promedia tiempos.

average_bin_frequencies = True
select_time = False
average_over_time = False

# PATHS

Ahora se presentan los paths si quieres ir en bucle con todos

In [ ]:
# ============================================================
# 1. CARPETA RAÍZ DE BESA CONNECTIVITY y
# ============================================================

besa_root = Path(r"G:\PROYECTO_SELF\self connectivity\BESACon")

# ============================================================
# 2. DICCIONARIO DE MÉTODOS DE CONECTIVIDAD
# ============================================================

connectivity_methods = {
    "coh": {
        "folder_prefix": "coh",
        "besa_folder": "Coherence",
        "label": "coherence",
    },
    "dpli": {
        "folder_prefix": "dpli",
        "besa_folder": "Directed Phase Lag Index",
        "label": "directed_phase_lag_index",
    },
    "wpli": {
        "folder_prefix": "wpli",
        "besa_folder": "Weighted Phase Lag Index",
        "label": "weighted_phase_lag_index",
    },
}



# ============================================================
# 3. MÉTODO TIME-FREQUENCY
# ============================================================

tf_method = "Wavelet Transform"


# ============================================================
# 4. COMBINACIONES POSIBLES MÉTODO × BANDA
# ============================================================

analysis_folders = {}

for conn_key, conn_info in connectivity_methods.items():
    for freq_key, freq_range in frequency_ranges.items():

        analysis_folder = (
            conn_info["folder_prefix"]
            + freq_key
        )

        analysis_folders[analysis_folder] = {
            "analysis_folder": analysis_folder,
            "connectivity_key": conn_key,
            "connectivity_folder": conn_info["besa_folder"],
            "connectivity_label": conn_info["label"],
            "freq_original": freq_key,
            "freq_range_hz": freq_range,
            "tf_method": tf_method,
        }
        
        
# ============================================================
# 5. OUTPUT
# ============================================================     


output_dir =  Path(r"G:\PROYECTO_SELF\self connectivity\exported_tables")
output_dir.mkdir(exist_ok=True)

## Selección de análisis a procesar

En este paso se define qué análisis de conectividad se van a procesar.

Cada tabla final se genera para una combinación concreta de:

- métrica de conectividad, por ejemplo `coh`, `dpli` o `wpli`;
- banda de frecuencia, por ejemplo `delta`, `theta`, `alpha`, `betalow`, `betahigh` o `gamma`.

Para cada combinación seleccionada, la tabla incluye los datos de **todas las condiciones experimentales** disponibles y de **todos los sujetos** encontrados dentro de las carpetas correspondientes.

Por ejemplo, si se selecciona `cohalpha`, se genera una única tabla para la métrica de conectividad `coh` en la banda `alpha`, incluyendo todas las condiciones experimentales y todos los sujetos.

In [ ]:
# ============================================================
# SELECCIÓN: UN SOLO ANÁLISIS
# ============================================================

# selected_analysis_folders = {
#     "cohalpha": analysis_folders["cohalpha"]
# }

# ============================================================
# SELECCIÓN: TODOS LOS ANÁLISIS
# ============================================================

selected_analysis_folders = analysis_folders

print(selected_analysis_folders.keys())

In [ ]:
# ============================================================
# MODO TEST / DEBUG
# ============================================================
# Para procesar todo, deja todo ONLY_ en None.


ONLY_ANALYSIS = None
ONLY_CONDITION = None
ONLY_CONN_SUBJECT = None

# Para sacar solo un análisis / condición / archivo, rellena ONLY_ el campo correspondiente.
#ONLY_ANALYSIS = "cohalpha"
# ONLY_CONDITION = "14"
# ONLY_CONN_SUBJECT = "s01b_vis_c_BV_Edit Markers_14_SNyDMNy ruidos (2).conn"


all_analysis_dfs = {}

for analysis_name, analysis_info in selected_analysis_folders.items():

    # ------------------------------------------------------------
    # TEST: procesar solo un análisis concreto
    # ------------------------------------------------------------
    if ONLY_ANALYSIS is not None and analysis_name != ONLY_ANALYSIS:
        continue

    print("\n====================================================")
    print("Procesando análisis:", analysis_name)
    print("====================================================")

    # Rutas del análisis
    analysis_dir = besa_root / analysis_info["analysis_folder"]
    connectivity_dir = analysis_dir / analysis_info["connectivity_folder"]
    tf_dir = connectivity_dir / analysis_info["tf_method"]

    if not tf_dir.exists():
        print("No existe:", tf_dir)
        continue

    # Condiciones experimentales
    condition_dirs = sorted([p for p in tf_dir.iterdir() if p.is_dir()])

    # ------------------------------------------------------------
    # TEST: procesar solo una condición concreta
    # ------------------------------------------------------------
    if ONLY_CONDITION is not None:
        condition_dirs = [
            p for p in condition_dirs
            if p.name == ONLY_CONDITION
        ]

    print("Condiciones encontradas:", [p.name for p in condition_dirs])

    analysis_final_dfs = []

    for condition_dir in condition_dirs:

        conn_files = sorted(condition_dir.glob("*.conn"))

        # ------------------------------------------------------------
        # TEST: procesar solo un archivo .conn concreto
        # ------------------------------------------------------------
        if ONLY_CONN_SUBJECT is not None:
            conn_files = [
                p for p in conn_files
                if p.name == ONLY_CONN_SUBJECT
            ]

        print("\nCondición:", condition_dir.name)
        print("Archivos .conn:", len(conn_files))

        for conn_file in conn_files:

            print("Procesando:", conn_file.name)

            # Leer archivo .conn
            header, channels, times, frequencies, conn_matrix = read_single_conn_file(
                conn_file=conn_file
            )

            # Convertir matriz a formato long
            df_long = conn_matrix_to_long_df(
                conn_file=conn_file,
                header=header,
                channels=channels,
                times=times,
                frequencies=frequencies,
                conn_matrix=conn_matrix,
                freq_original_folder=analysis_info["analysis_folder"],
            )

            del conn_matrix, channels, times, frequencies
            

            # Limpiar diagonal y filtrar banda de frecuencia
            df_clean = clean_conn_long_df(
                df=df_long,
                frequency_ranges=frequency_ranges,
                inclusive_bands=inclusive_bands,
            )
            del df_long


            # Seleccionar/promediar valores finales
            df_avg_values = average_values_df(
                df=df_clean,
                average_bin_frequencies=average_bin_frequencies,
                select_time=select_time,
                average_over_time=average_over_time,
            )
            del df_clean
            
            # Crear columnas descriptivas de condición
            df_changed_columns = changing_columns(
                df=df_avg_values,
                condition_mapping=condition_mapping,
            )
            del df_avg_values


            analysis_final_dfs.append(df_changed_columns)

    if len(analysis_final_dfs) == 0:
        print("No se generó tabla para:", analysis_name)
        continue

    # Unir todas las condiciones y sujetos del análisis
    df_analysis_final = pd.concat(
        analysis_final_dfs,
        ignore_index=True
    )


    # Exportar CSV
    output_file = output_dir / f"{analysis_name}_all_conditions_clean_long.csv"

    df_analysis_final.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )
    print("\nTabla final:", analysis_name)
    print(df_analysis_final.shape)
    print("CSV exportado en:", output_file)
    del df_analysis_final
    



In [ ]:
# df_long 


In [ ]:
# Limpiar diagonal y filtrar banda de frecuencia
# df_clean 


In [ ]:

# df_avg_values 


In [ ]:
            
# Crear columnas descriptivas de condición
# df_changed_columns 


In [ ]:
# df_analysis_final